# Clase 11 — Clustering
## Segmentamos a los clientes de un distribuidor mayorista: ¿en quién invertimos?

---

Somos el **equipo comercial de un distribuidor mayorista** de alimentos y productos de limpieza. Les vendemos a restaurantes, hoteles, cafés, almacenes y supermercados. Este año nos aprobaron un presupuesto de **US$ 600.000** para mejorar la atención comercial, y no alcanza para atender a todos igual. Nos preguntan:

> ¿Qué tipos de cliente tenemos, y en cuáles conviene invertir?

**El problema:** tenemos 440 clientes y cuánto compró cada uno en seis categorías durante el último año, pero ninguna etiqueta que diga "este es de tal tipo". Nadie nos dijo cuántos tipos hay ni cómo se llaman. Y acá está el cambio respecto de las Clases 05 a 10: **no hay variable objetivo**. No predecimos nada, descubrimos estructura.

**Hipótesis:** hay grupos de clientes con patrones de compra muy distintos, y cada grupo pide una atención comercial distinta.

**Datos:** son **reales**: [Wholesale Customers](https://archive.ics.uci.edu/dataset/292/wholesale+customers), UCI Machine Learning Repository (Cardoso, 2013, licencia CC BY 4.0). Es un distribuidor mayorista de Portugal, pero el problema es el mismo que el de cualquier distribuidor de acá. El dataset no aclara la moneda (habla de "unidades monetarias"), así que para simplificar lo expresamos en **dólares**. Traducimos los nombres de las columnas al español.

**Esta clase la resolvemos entre todos.** El notebook corre los números, pero las tres decisiones importantes las toma la clase en el pizarrón:

1. **¿Cuántos grupos hacemos?** (el K)
2. **¿Cómo se llama cada grupo?** (los arquetipos)
3. **¿Qué hacemos con los US$ 600.000?** (el presupuesto)

Y cerramos con una **matriz de arquetipos** de dos ejes, que dibujamos en el pizarrón.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
COLORES = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860", "#DA8BC3", "#8C8C8C"]

---
## 1. Carga y vistazo

Antes de agrupar, miramos qué mide cada columna y en qué magnitudes. Eso va a definir si podemos usar los datos tal cual o hay que prepararlos.

| Columna | Qué representa |
|---|---|
| `id_cliente` | identificador anónimo (no entra al modelo) |
| `canal` | `Horeca` (hoteles, restaurantes, cafés) o `Retail` (tiendas y supermercados) |
| `region` | Lisboa, Oporto u Otra |
| `frescos` | gasto anual en productos frescos |
| `lacteos` | gasto anual en lácteos |
| `almacen` | gasto anual en almacén (abarrotes) |
| `congelados` | gasto anual en congelados |
| `limpieza_papel` | gasto anual en detergentes y papel |
| `delicatessen` | gasto anual en delicatessen |

Los seis gastos están en dólares. **`canal` y `region` no los vamos a usar para agrupar.** `canal` lo guardamos para el final, para ver si el algoritmo, sin conocerlo, descubre algo parecido.

In [ ]:
clientes = pd.read_csv("clientes_mayoristas.csv")

GASTOS = ["frescos", "lacteos", "almacen", "congelados", "limpieza_papel", "delicatessen"]

print(f"{clientes.shape[0]} clientes | {len(GASTOS)} categorías de gasto")
clientes[GASTOS].describe().round(0)

**Mirá la fila `mean` contra la fila `50%` (la mediana), y después la fila `max`.** En todas las categorías el promedio le gana a la mediana, y el máximo es muchísimas veces la mediana: en `frescos`, la mediana es de unos US$ 8.500 y el cliente más grande compró más de US$ 112.000. Hay muchos clientes chicos y unos pocos gigantes. Guardate esa observación, porque en la sección 3 va a ser un problema para el algoritmo.

---
## 2. EDA: ¿cómo se distribuye el gasto?

Antes de usar un algoritmo, miremos los datos. Un histograma por categoría nos dice si los clientes se parecen entre sí o si hay mucha dispersión.

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(13, 6))
for ax, columna in zip(ejes.flatten(), GASTOS):
    ax.hist(clientes[columna], bins=30, color="#4C72B0")
    ax.set_title(columna)
plt.tight_layout()
plt.show()

**En las seis categorías pasa lo mismo:** una pila enorme de clientes que compran poco, y una cola larga hacia la derecha con unos pocos que compran muchísimo. Es el patrón típico de cualquier negocio de clientes: pocos clientes grandes, muchos chicos. Veamos qué tanto pesan los grandes en las ventas:

In [ ]:
clientes["gasto_total"] = clientes[GASTOS].sum(axis=1)

diez_por_ciento = round(len(clientes) * 0.10)
ventas_top = clientes["gasto_total"].nlargest(diez_por_ciento).sum()
porcentaje = ventas_top / clientes["gasto_total"].sum()

print(f"El 10% de los clientes ({diez_por_ciento}) concentra el {porcentaje:.0%} de las ventas")

**Los grandes importan, pero no son todo el negocio:** el 10% de los clientes explica casi el 28% de las ventas, o sea que el 72% restante viene de la masa de clientes medianos y chicos. Si diseñamos una estrategia comercial mirando solo a los gigantes, nos olvidamos de la mayor parte de la facturación.

A ojo, con histogramas de a una variable, no vemos si hay **tipos** de cliente. Para eso necesitamos que el algoritmo mire las seis categorías juntas.

---
## 3. Preparación: escalar y domar los extremos

K-Means decide a qué grupo va cada cliente midiendo **distancias**. Es el mismo problema que tuvimos con KNN en la **Clase 05**: si una variable tiene valores de decenas de miles y otra de cientos, la distancia la define casi entera la de los valores grandes. Acá todas las categorías están en dólares, pero con magnitudes muy distintas (`frescos` promedia unas 12.000; `delicatessen`, unas 1.500).

La solución estándar es **estandarizar**: pasar cada variable a "cuántos desvíos por encima o por debajo del promedio está". Después de eso, todas pesan parecido.

> A diferencia de CART (Clase 07), que no necesitaba escalar porque corta variable por variable, acá el escalado **no es opcional**.

Hagamos la prueba con esto solo, como lo venimos haciendo: estandarizar y correr K-Means con 4 grupos (el 4 es solo para esta demo; el K de verdad lo elige la clase en la sección 4).

In [ ]:
X = clientes[GASTOS]

escalador_simple = StandardScaler()
X_solo_escalado = escalador_simple.fit_transform(X)

intento = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_solo_escalado)

tamanos_intento = pd.Series(intento.labels_).value_counts().sort_index()
tamanos_intento.index.name = "cluster"
tamanos_intento.name = "clientes"
tamanos_intento

**Mirá los tamaños.** Dos de los cuatro grupos tienen apenas 10 y 5 clientes, y uno solo concentra a 315 de los 440. El algoritmo gastó la mitad de sus grupos en aislar a 15 clientes extremos. Veamos quiénes son los de los grupos más chicos:

In [ ]:
chicos = tamanos_intento[tamanos_intento < 20].index
clientes[np.isin(intento.labels_, chicos)][["id_cliente"] + GASTOS].sort_values("frescos", ascending=False)

Son clientes enormes: algunos compran decenas de miles de dólares en una sola categoría. **No son errores de carga**, son clientes reales y muy valiosos. Pero para nuestra decisión, un "grupo" de 5 clientes no sirve: no podemos armar una estrategia de presupuesto para un grupo que se resuelve llamando por teléfono a cada uno, y mientras tanto **315 clientes quedaron mezclados en un solo bloque**.

Esto es lo que dice el PDF sobre la limitación de K-Means con **outliers**: el centroide es un promedio, y un promedio se deja arrastrar por los extremos.

**Una trampa conceptual.** Venimos de CART (Clase 07), donde los valores extremos eran parte de la señal y no se tocaban. Y de la Clase 04 (regresión lineal), donde el reflejo era sacarlos. Acá ninguna de las dos respuestas sirve: **sacarlos sería sacar a nuestros mejores clientes**, y dejarlos tal cual rompe el agrupamiento. La salida es una tercera: **transformar la escala** para que un cliente que compra 10 veces más no esté a 10 veces más de distancia.

### La transformación logarítmica

Aclaración: **esto no está en el PDF de teoría**; lo agregamos porque con datos reales de gasto el escalado solo no alcanza, como acabamos de ver.

**La idea en una frase:** el logaritmo cambia la pregunta. En vez de medir *cuántos dólares más* compra un cliente, mide *cuántas veces más* compra.

Para verlo, tomemos cuatro compras anuales: dos de clientes chicos y dos de clientes grandes. En cada par, el segundo compra **exactamente el doble** que el primero.

In [ ]:
ejemplo = pd.DataFrame({"compra anual (US$)": [1_000, 2_000, 50_000, 100_000]})
ejemplo["logaritmo (base 10)"] = np.log10(ejemplo["compra anual (US$)"]).round(2)
ejemplo

**Fijate en las diferencias.** En dólares, la distancia entre los dos clientes chicos es de 1.000 y entre los dos grandes es de 50.000: **50 veces más**. Para K-Means, que trabaja con distancias, los dos grandes quedan en otro planeta, aunque en proporción hicieron lo mismo (duplicaron la compra). En la columna del logaritmo, las dos diferencias valen lo mismo (0,3): **el logaritmo cuenta cuántas veces se multiplica la compra, no cuántos dólares se suman**. Cada punto que sube la columna equivale a multiplicar por 10: de 1.000 (3,0) a 10.000 (4,0) y a 100.000 (5,0).

Tres aclaraciones para que no queden dudas:

- **No cambia el orden.** El cliente más grande sigue siendo el más grande; lo único que cambia es qué tan lejos queda de los demás.
- **En el código usamos `np.log1p`**, que es el logaritmo natural de (1 + compra). El "+1" evita problemas si algún cliente compra cero. Que la base sea 10 o natural cambia la escala del número, no la idea.
- **Al final se deshace.** Los centroides los pasamos de nuevo a dólares (sección 6) para poder hablar en plata y no en logaritmos.

Es lo que se hace habitualmente cuando el gasto o los ingresos tienen la forma de cola larga que vimos en los histogramas.

In [ ]:
X_log = np.log1p(X)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(X["frescos"], bins=30, color="#4C72B0")
ejes[0].set_title("frescos (US$)")
ejes[1].hist(X_log["frescos"], bins=30, color="#55A868")
ejes[1].set_title("frescos (logaritmo)")
plt.show()

**A la izquierda, la pila y la cola larga. A la derecha, una distribución mucho más simétrica:** los clientes chicos y los medianos se separan entre sí y los gigantes quedan en la punta, pero ya no tan lejos del resto. Ahora sí estandarizamos, sobre los datos con logaritmo.

In [ ]:
escalador = StandardScaler()
X_esc = pd.DataFrame(escalador.fit_transform(X_log), columns=GASTOS)

X_esc.describe().loc[["mean", "std"]].round(2)

Las seis variables quedaron con promedio 0 y desvío 1. Dos pasos en orden: **primero logaritmo, después estandarizar**. De acá en adelante, todo el análisis usa `X_esc`.

---
## 4. Modelo: ¿cuántos grupos hacemos?

K-Means hace lo que vimos en la teoría: pone K centroides, asigna cada cliente al más cercano, recalcula el centroide como el promedio de su grupo y repite hasta que dejan de moverse. Lo único que **no** puede hacer solo es decidir **cuántos grupos queremos**. Eso lo decimos nosotros antes de correrlo.

Como no hay conjunto de test que nos diga "acertaste", usamos la **inercia**: qué tan compactos quedan los grupos. Probamos K de 1 a 10 y miramos dónde agregar un grupo más ya no mejora demasiado (el codo).

> Una aclaración fina: la inercia que calcula sklearn suma las distancias **al cuadrado** de cada cliente a su centroide. Es un detalle técnico, la lectura es la misma: cuanto más baja, más compactos los grupos. Y siempre baja al agregar grupos (con un grupo por cliente sería cero), por eso **no buscamos la inercia mínima, buscamos el codo**.

In [ ]:
resultados = []
for k in range(1, 11):
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_esc)
    resultados.append({"K": k, "inercia": modelo.inertia_})

codo = pd.DataFrame(resultados)
codo["mejora_vs_K_anterior_%"] = (-codo["inercia"].pct_change() * 100).round(1)
codo["inercia"] = codo["inercia"].round(0)
codo

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(codo["K"], codo["inercia"], marker="o", color="#4C72B0")
ax.set_xticks(range(1, 11))
ax.set_xlabel("K (cantidad de grupos)")
ax.set_ylabel("inercia")
plt.show()

**¿Dónde está el codo?** Acá es mucho menos nítido que en los ejemplos de manual. La curva cae fuerte hasta K=2 (30%), baja un poco menos hasta K=3 (16%) y K=4 (11%), y después las mejoras se van achicando de a poco (9%, 7%...) sin un quiebre claro. **K entre 2 y 5 es razonable (por eso votamos 3, 4 y 5 en el pizarrón; el 2 es tan grueso que casi no sirve para tomar decisiones), y de ahí en más empezamos a partir grupos que quizás se tratan igual.** Así se ven los datos reales: la curva nos acota el rango, pero **la última palabra la tiene el negocio**.

### Pausa: pizarrón 1 — ¿Qué K elegimos?

Escribimos en el pizarrón K = 3, 4 y 5 y votamos. Para decidir, no vale decir "el codo dice...". Pensemos en negocio:

- ¿Cuántas **estrategias comerciales distintas** podemos sostener con US$ 600.000? Cada grupo nuevo es una acción nueva.
- Si hacemos pocos grupos, ¿mezclamos clientes que necesitan cosas diferentes?
- Si hacemos muchos, ¿estamos partiendo un grupo en dos que en la práctica se atienden igual?

Con la decisión tomada, la escribimos en la variable `K` de la celda que sigue a la próxima. Antes, una advertencia sobre un límite del algoritmo.

**Otro límite: la inicialización.** K-Means arranca con centroides al azar, y según dónde caigan puede terminar en lugares distintos. Corramos el mismo modelo (con 4 grupos, otra vez solo para la demo) 8 veces, cada una con un arranque aleatorio distinto y **una sola** pasada:

In [ ]:
for semilla in range(8):
    prueba = KMeans(n_clusters=4, init="random", n_init=1, random_state=semilla).fit(X_esc)
    print(f"arranque {semilla}: inercia = {prueba.inertia_:,.0f}")

**Mismos datos, mismo K, y la inercia cambia según el arranque**, aunque en este dataset el efecto es moderado (entre 1.387 y 1.444, unos 4% de diferencia). Con grupos más enredados o con otro K puede ser mucho mayor. La salida práctica no cuesta nada: `n_init=10`, que repite todo 10 veces desde puntos distintos y **se queda con la mejor corrida**. Es lo que venimos usando en todo el notebook y lo que usaríamos siempre en un caso real.

In [ ]:
K = 4  # lo decide la clase en el pizarrón

modelo = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X_esc)
clientes["cluster"] = modelo.labels_

---
## 5. Perfilado: ¿quiénes son los clientes de cada grupo?

El algoritmo nos devolvió un número (0, 1, 2...) para cada cliente. Ese número **no significa nada** por sí solo: no sabe que un grupo es "los que compran mucho fresco". Interpretarlo es trabajo nuestro, y es la parte que le da valor al análisis.

Y como no hay etiqueta real contra la cual comparar, **no existe un "accuracy" del clustering**. Un buen agrupamiento es el que nos permite **tomar una decisión distinta para cada grupo**. Si dos grupos se atienden igual, sobra uno.

Empecemos por cuántos clientes hay en cada uno:

In [ ]:
tamanos = clientes["cluster"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(tamanos.index.astype(str), tamanos.values, color=COLORES[:K])
ax.set_xlabel("cluster")
ax.set_ylabel("clientes")
plt.show()

Con el logaritmo, **los grupos quedaron razonablemente parejos** (todos entre el 14% y el 30% de los clientes). Ahora sí podemos pensar una estrategia para cada uno. Veamos **qué hace cada grupo distinto del promedio**. En el mapa de calor, cada celda dice cuántos desvíos por encima (rojo) o por debajo (azul) del cliente promedio está el centroide del grupo en esa categoría. Se lee por fila: una fila = el retrato de un grupo.

In [ ]:
centros = pd.DataFrame(modelo.cluster_centers_, columns=GASTOS)

fig, ax = plt.subplots(figsize=(9, 3.5))
sns.heatmap(centros, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("cluster")
plt.xticks(rotation=30, ha="right")
plt.show()

El mapa de calor sirve para **ver** el patrón, pero está en desvíos estándar de la escala logarítmica. Para hablar con el negocio necesitamos los números en dólares, sin transformar: cuánto compra en promedio un cliente de cada grupo, y cuánto pesa cada grupo en las ventas.

In [ ]:
perfil = clientes.groupby("cluster")[GASTOS + ["gasto_total"]].mean().round(0)
perfil.insert(0, "clientes", tamanos)

ventas_por_grupo = clientes.groupby("cluster")["gasto_total"].sum()
perfil["% de las ventas"] = (ventas_por_grupo / ventas_por_grupo.sum() * 100).round(0)
perfil

**Cada fila es el cliente "típico" de un grupo**, y la última columna agrega algo que no estaba en los histogramas: **cuánto pesa cada grupo en la facturación**. Leyendo la tabla y el mapa de calor juntos aparecen perfiles bien distintos: quién compra sobre todo frescos y congelados, quién compra mucho almacén, lácteos y limpieza, y quién compra poco de todo. Y el grupo más grande en clientes no es el que más factura.

Ahora, una pregunta que en clustering no tiene una respuesta única pero sí un buen **chequeo de sentido común**. En el dataset hay una columna que el algoritmo **nunca vio**: `canal` (Horeca o Retail). ¿Los grupos que encontró tienen que ver con eso?

In [ ]:
pd.crosstab(clientes["cluster"], clientes["canal"])

**Sin que nadie se lo dijera, el algoritmo separó en buena medida a los clientes de gastronomía (Horeca) de los de tiendas y supermercados (Retail):** dos de los grupos son casi todo Horeca y los otros dos son mayoritariamente Retail. Esto **no es un accuracy**, porque el objetivo nunca fue adivinar el canal, pero es una buena señal de que los grupos reflejan algo real del negocio.

Y fijate lo que **no** nos dice el canal: los clientes Horeca se parten en **dos grupos muy distintos** (uno compra mucho fresco y congelados, el otro compra poco de todo). Si hubiéramos segmentado solo por canal, habríamos tratado igual a dos tipos de cliente que piden cosas distintas. Ese es el valor extra del clustering.

### Pausa: pizarrón 2 — ¿Cómo se llama cada grupo?

Un nombre por grupo, escrito en el pizarrón, como en el ejemplo de streaming de la teoría ("Usuarios comprometidos", "Usuarios en riesgo de churn"). Para cada grupo:

- ¿Qué compra? Una frase.
- ¿Qué nombre le pondrían para que lo entienda alguien de otra área que no vio este análisis?
- ¿Es un grupo que queremos **retener**, **hacer crecer** o **atender con menos costo**?

Con los nombres definidos, los cargamos acá abajo.

In [ ]:
# La clase completa estos nombres en el pizarrón. Si no se completan, queda "Cluster 0", "Cluster 1"...
NOMBRES = {}
for c in range(K):
    NOMBRES[c] = f"Cluster {c}"

# Ejemplo de cómo se completa (reemplazar según lo que decida la clase):
# NOMBRES = {0: "...", 1: "...", 2: "...", 3: "..."}

---
## 6. La matriz de arquetipos

Tenemos seis categorías y K grupos, pero una reunión de directivos no se maneja con tablas de seis columnas. Necesitamos **una imagen**: una matriz de **dos ejes** donde cada arquetipo ocupe un lugar y se vea de un vistazo en qué se parece y en qué se diferencia de los otros.

La decisión clave es **qué dos categorías ponemos en los ejes**. No hay una correcta: depende de la pregunta de negocio. Las líneas punteadas marcan la mediana de cada variable y dividen el plano en cuatro cuadrantes. Las X grandes son los centroides: el cliente "típico" de cada arquetipo, tal como lo ubicó el algoritmo.

Un detalle de lectura: **los dos ejes están en escala logarítmica**, por la misma razón de la sección 3. Si no, casi todos los clientes quedarían amontonados en la esquina de abajo a la izquierda y los gigantes estirarían el gráfico. En esta escala, cada línea de la grilla vale 10 veces la anterior.

In [ ]:
# centroides del modelo, devueltos a dólares (deshacemos el escalado y el logaritmo)
centros_reales = pd.DataFrame(np.expm1(escalador.inverse_transform(modelo.cluster_centers_)), columns=GASTOS)


def etiqueta(columna):
    return columna.replace("_", " ")


def dibujar_matriz(datos, centros_ejes, eje_x, eje_y, nombres):
    fig, ax = plt.subplots(figsize=(10, 6.5))
    for c in sorted(datos["cluster"].unique()):
        grupo = datos[datos["cluster"] == c]
        ax.scatter(grupo[eje_x], grupo[eje_y], s=25, alpha=0.5, color=COLORES[c], label=nombres[c])
        ax.scatter(centros_ejes.loc[c, eje_x], centros_ejes.loc[c, eje_y], s=300, marker="X", color=COLORES[c], edgecolor="black", linewidth=1.5)
    ax.axvline(datos[eje_x].median(), color="gray", linestyle="--")
    ax.axhline(datos[eje_y].median(), color="gray", linestyle="--")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(etiqueta(eje_x) + " (US$, escala logarítmica)")
    ax.set_ylabel(etiqueta(eje_y) + " (US$, escala logarítmica)")
    ax.legend(title="Arquetipo", loc="upper left")
    plt.show()


def tabla_cuadrantes(datos, centros_ejes, eje_x, eje_y, nombres):
    corte_x = datos[eje_x].median()
    corte_y = datos[eje_y].median()
    filas = []
    for c in sorted(datos["cluster"].unique()):
        valor_x = centros_ejes.loc[c, eje_x]
        valor_y = centros_ejes.loc[c, eje_y]
        if valor_x > corte_x:
            nivel_x = "alto"
        else:
            nivel_x = "bajo"
        if valor_y > corte_y:
            nivel_y = "alto"
        else:
            nivel_y = "bajo"
        filas.append({"arquetipo": nombres[c], etiqueta(eje_x): round(valor_x), etiqueta(eje_y): round(valor_y),
                      "nivel en X": nivel_x, "nivel en Y": nivel_y})
    return pd.DataFrame(filas)

In [ ]:
EJE_X = "almacen"  # la clase puede cambiar estos dos ejes y volver a correr
EJE_Y = "delicatessen"

dibujar_matriz(clientes, centros_reales, EJE_X, EJE_Y, NOMBRES)
tabla_cuadrantes(clientes, centros_reales, EJE_X, EJE_Y, NOMBRES)

**Ojo con cómo leer este gráfico.** Los puntos se mezclan bastante porque dos ejes no capturan todo lo que el algoritmo vio en seis categorías, y en los datos reales los grupos **no** están separados por huecos: son nubes que se pisan. Lo que se lee son las **X** (los clientes "típicos"): con **almacén** y **delicatessen**, las cuatro X (K = 4) caen en cuadrantes distintos. Un cuadrante es el de las compras grandes de almacén con poca delicatessen, otro es el de mucha delicatessen y poco almacén, otro tiene las dos altas y el último las dos bajas.

### Pausa: pizarrón 3 — Dibujamos la matriz

Cada uno de los arquetipos que nombramos va en un cuadrante. En el pizarrón: dos ejes, cuatro casilleros, los nombres adentro. Después, dos preguntas para discutir:

- **¿Cambia la matriz si cambiamos los ejes?** Prueben con `frescos` y `limpieza_papel`, o con `lacteos` y `congelados`, y vuelvan a correr la celda. Los grupos son los mismos, pero se ven distinto, y no todos los pares separan a los cuatro.
- **¿Hay algún cuadrante vacío?** Con los ejes por defecto y K = 4 están todos ocupados, así que prueben con otros ejes o con otro K. Si aparece un cuadrante vacío, ¿es un hueco en los datos o una oportunidad de negocio (un tipo de cliente que no estamos captando)?

---
## 7. Decisión con presupuesto

Llegamos a la parte que importa. Tenemos **US$ 600.000** y la dirección nos pasó un menú de acciones posibles. Suman bastante más que el presupuesto, así que **hay que elegir**. Cada acción apunta a un tipo de comportamiento de compra; a nosotros nos toca decidir a qué arquetipo se la asignamos.

In [ ]:
PRESUPUESTO = 600_000

acciones = pd.DataFrame({
    "acción": [
        "Entrega diaria de frescos con logística refrigerada",
        "Descuento por volumen en almacén, lácteos y limpieza",
        "Campaña de productos premium (delicatessen)",
        "Ejecutivo de cuenta dedicado",
        "Catálogo digital y pedido online automático",
        "Visita comercial para ampliar las categorías que compra",
    ],
    "costo": [250_000, 200_000, 100_000, 300_000, 80_000, 150_000],
    "apunta a": [
        "clientes que compran sobre todo frescos y congelados",
        "clientes con compras grandes de almacén, lácteos y limpieza",
        "clientes que ya compran productos premium",
        "clientes de gasto total alto",
        "clientes chicos, con gasto bajo en todas las categorías",
        "clientes que compran pocas categorías",
    ],
}, index=[1, 2, 3, 4, 5, 6])
acciones.index.name = "id"


def formato_usd(valor):
    return "US$ " + f"{valor:,.0f}".replace(",", ".")


mostrar = acciones.copy()
mostrar["costo"] = mostrar["costo"].apply(formato_usd)
print(f"Presupuesto: {formato_usd(PRESUPUESTO)} | Si hiciéramos todo: {formato_usd(acciones['costo'].sum())}")
mostrar

**Tres números para pensar antes de elegir.** Primero, el **costo por cliente alcanzado**: la misma acción sale muy distinto según el tamaño del grupo al que le llega. Segundo, el **gasto anual promedio** del grupo, que sí es una medida de valor real (acá tenemos las compras de cada cliente, no una aproximación). Tercero, cuánto representa ese costo **como porcentaje de lo que el cliente ya nos compra**: invertir US$ 1.000 por cliente en alguien que compra 50.000 por año no es lo mismo que en alguien que compra 15.000. Ojo con una limitación: el costo por cliente supone que todo el grupo usa la acción, cosa que en la realidad no pasa. El notebook calcula las cuentas; **la decisión es de la clase**.

La decisión se escribe como un diccionario `{id de la acción: número de cluster al que se la asignamos}`. El ejemplo de abajo es solo para mostrar cómo funciona; la clase lo reemplaza con lo que decida en el pizarrón.

In [ ]:
def resumen_decision(decision):
    filas = []
    gastado = 0
    for id_accion, cluster in decision.items():
        costo = acciones.loc[id_accion, "costo"]
        alcanzados = (clientes["cluster"] == cluster).sum()
        gasto_grupo = clientes.loc[clientes["cluster"] == cluster, "gasto_total"].mean()
        costo_por_cliente = costo / alcanzados
        filas.append({
            "acción": acciones.loc[id_accion, "acción"],
            "arquetipo": NOMBRES[cluster],
            "costo": formato_usd(costo),
            "clientes alcanzados": alcanzados,
            "costo por cliente": formato_usd(costo_por_cliente),
            "gasto anual promedio del grupo": formato_usd(gasto_grupo),
            "costo / gasto anual": f"{costo_por_cliente / gasto_grupo:.1%}",
        })
        gastado += costo
    saldo = PRESUPUESTO - gastado
    print(f"Gastado: {formato_usd(gastado)} de {formato_usd(PRESUPUESTO)} | Saldo: {formato_usd(saldo)}")
    if saldo < 0:
        print("Nos pasamos del presupuesto: hay que sacar o cambiar alguna acción.")
    return pd.DataFrame(filas)


# Ejemplo con la numeración de K = 4: acción 1 para el cluster 1 y acción 2 para el cluster 3.
# Si la clase eligió otro K, los números de cluster cambian: hay que mirar la tabla de la sección 5.
DECISION = {1: 1, 2: 3}

resumen_decision(DECISION)

### Pausa: pizarrón 4 — Repartimos los US$ 600.000

Armamos la decisión en el pizarrón y la cargamos en `DECISION`. Algunas preguntas para empujar la discusión:

- ¿Repartimos **un poco a todos** o **mucho a pocos**? ¿Qué se pierde con cada opción?
- ¿Cuál es el grupo **más caro de alcanzar por cliente**? ¿Vale la pena?
- ¿Invertimos en los que **ya compran mucho** (los blindamos) o en los que **compran poco** (los desarrollamos)? ¿Qué es más rentable? Nadie tiene la respuesta correcta, pero sí hay que **defender** la elección con un número del resumen.
- Un mismo grupo puede recibir más de una acción: ¿conviene concentrar o repartir? (Probá poniendo dos acciones sobre el mismo cluster en `DECISION`.) Ojo con las claves del diccionario: cada acción aparece una sola vez.
- ¿Qué pasa si el presupuesto fuera 300.000 en vez de 600.000? Cambien `PRESUPUESTO` y vuelvan a correr.

---
## 8. Cierre

**Qué hicimos.** Partimos de 440 clientes sueltos y seis columnas de gasto, y terminamos con una matriz de arquetipos y un presupuesto repartido con criterio. El algoritmo hizo una sola cosa: **agrupar**. Todo lo demás lo decidimos nosotros.

**Qué llevarse:**

- **K, los ejes de la matriz y el presupuesto son decisiones de negocio**, no del algoritmo. La curva del codo acota el rango de K, pero no lo define.
- **Con datos reales, escalar no alcanza.** Si el gasto tiene cola larga, K-Means dedica grupos enteros a un puñado de clientes extremos. Primero logaritmo, después estandarizar.
- **Los extremos no se eliminan ni se ignoran.** Eran nuestros mejores clientes. Se tratan con una transformación, no con una tijera.
- **No hay accuracy en clustering.** Un buen agrupamiento es el que permite tomar una decisión distinta para cada grupo. Ver si los grupos se parecen a una variable que el algoritmo no usó (el canal) es un chequeo de sentido común, no un puntaje.
- **Los números de cluster no significan nada hasta que los interpretamos.** El valor está en el perfilado y en los nombres.

**Qué vigilar con K-Means:**

- Elegir K no es obvio: dos personas razonables pueden quedarse con 3 o con 4.
- El resultado depende del arranque: usar siempre `n_init` alto.
- Los valores extremos arrastran los centroides: hay que mirarlos antes de agrupar.

**Gancho a la próxima clase.** Hoy tuvimos que elegir **dos** categorías para dibujar la matriz, aunque el algoritmo usó **seis**. ¿Y si pudiéramos resumir esas seis variables en dos nuevas, que conserven lo más importante de la información, y graficar a todos los clientes de una? Eso es **PCA**, la Clase 12.